# Exploration du Dataset — Amazon Food Reviews

## Contexte
Les entreprises reçoivent des milliers d'avis clients
mais n'ont pas le temps de les lire un par un.  
Ce projet automatise l'analyse de sentiment de ces avis.

## Ce notebook
Dans ce notebook on va :
1. Charger le dataset Amazon Food Reviews
2. Explorer sa structure et ses colonnes
3. Créer les labels de sentiment à partir des notes
4. Sauvegarder le résultat pour la prochaine étape

## 1. Chargement des librairies

In [1]:
import pandas as pd

print("Librairies chargées avec succès")

Librairies chargées avec succès


## 2. Chargement du dataset

In [2]:
df = pd.read_csv("../data/Reviews.csv")

# Afficher le nombre de lignes et de colonnes
print(f"Nombre d'avis : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")

Nombre d'avis : 568454
Nombre de colonnes : 10


## 3. Exploration des colonnes


In [3]:
df.head(5)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## Explication des colonnes

| Colonne | Description | Utile ? |
|---|---|---|
| `Id` | Numéro unique de l'avis | ❌ |
| `ProductId` | Identifiant du produit | ❌ |
| `UserId` | Identifiant de l'utilisateur | ❌ |
| `ProfileName` | Nom de l'utilisateur | ❌ |
| `HelpfulnessNumerator` | Nombre de personnes qui ont trouvé l'avis utile | ❌ |
| `HelpfulnessDenominator` | Nombre de personnes qui ont voté | ❌ |
| `Score` | Note de 1 à 5 → servira de label de sentiment | ✅ |
| `Time` | Date de l'avis (format timestamp) | ❌ |
| `Summary` | Titre court de l'avis | ⚠️ |
| `Text` | Texte complet de l'avis → notre donnée principale | ✅ |

**Colonnes retenues** : `Text` et `Score` uniquement.  
`Summary` pourra être combiné à `Text` dans une version améliorée du projet.

## 4. Vérification de la qualité des données

Avant d'utiliser les données on va d'abord vérifier deux choses :
- Y a-t-il des valeurs manquantes ?
- Y a-t-il des doublons ?

In [4]:
# Vérifier les valeurs manquantes dans chaque colonne
print("=== Valeurs manquantes ===")
print(df[["Text", "Score"]].isnull().sum())

# Vérifier les doublons
print(f"\n=== Doublons ===")
print(f"Nombre de doublons : {df.duplicated().sum()}")

=== Valeurs manquantes ===
Text     0
Score    0
dtype: int64

=== Doublons ===
Nombre de doublons : 0


- Aucune valeur manquante dans `Text` et `Score`
- Aucun doublon détecté

Les données sont propres, on peut continuer sans nettoyage supplémentaire.

## 5. Répartition des notes

On analyse combien d'avis il y a par note.
C'est important parce que si on a beaucoup plus de 5 étoiles que de 1 étoile, notre modèle va avoir du mal à apprendre à reconnaître les avis négatifs (déséquilibre entre les classes positives et négatives).

In [5]:
# Compter le nombre d'avis par note
repartition = df["Score"].value_counts().sort_index()

# Afficher en nombre et en pourcentage
for note, nombre in repartition.items():
    pourcentage = (nombre / len(df)) * 100
    print(f"Note {note} : {nombre:>6} avis ({pourcentage:.1f}%)")

Note 1 :  52268 avis (9.2%)
Note 2 :  29769 avis (5.2%)
Note 3 :  42640 avis (7.5%)
Note 4 :  80655 avis (14.2%)
Note 5 : 363122 avis (63.9%)


### Analyse des résultats

| Note | Nombre | Pourcentage |
|---|---|---|
| 1 étoile | 52 268 | 9.2% |
| 2 étoiles | 29 769 | 5.2% |
| 3 étoiles | 42 640 | 7.5% |
| 4 étoiles | 80 655 | 14.2% |
| 5 étoiles | 363 122 | 63.9% |

### Problème détecté : déséquilibre des classes

63.9% des avis sont des 5 étoiles.  
Si on entraîne un modèle sur ces données telles quelles,
il va apprendre à dire "positif" presque tout le temps
et avoir quand même 63% de précision sans vraiment apprendre.

**Solution** : on équilibrera les classes dans la suite de notre analyse.

## 6. Création des labels de sentiment

On transforme les notes (1 à 5) en sentiment (positif/negatif/neutre).
C'est le "label" qu'on va donner au modèle pour qu'il apprenne.

| Note | Sentiment |
|---|---|
| 4 ou 5 | positif |
| 1 ou 2 | negatif |
| 3 | neutre |

In [6]:
# Fonction qui convertit une note en sentiment
def score_vers_sentiment(score):
    if score >= 4:
        return "positif"
    elif score <= 2:
        return "negatif"
    else:
        return "neutre"

# Appliquer la fonction sur toute la colonne Score
df["sentiment"] = df["Score"].apply(score_vers_sentiment)

# Vérifier le résultat
print("=== Répartition des sentiments ===")
for sentiment, nombre in df["sentiment"].value_counts().items():
    pourcentage = (nombre / len(df)) * 100
    print(f"{sentiment:>8} : {nombre:>6} avis ({pourcentage:.1f}%)")

# Afficher un exemple de chaque sentiment
print("\n=== Exemples ===")
for sentiment in ["positif", "negatif", "neutre"]:
    exemple = df[df["sentiment"] == sentiment]["Text"].iloc[0]
    print(f"\n{sentiment.upper()} : {exemple[:100]}...")

=== Répartition des sentiments ===
 positif : 443777 avis (78.1%)
 negatif :  82037 avis (14.4%)
  neutre :  42640 avis (7.5%)

=== Exemples ===

POSITIF : I have bought several of the Vitality canned dog food products and have found them all to be of good...

NEGATIF : Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. No...

NEUTRE : This seems a little more wholesome than some of the supermarket brands, but it is somewhat mushy and...


### Résultats

- **positif** : 443 777 avis (78.1%)
- **negatif** :  82 037 avis (14.4%)
- **neutre**  :  42 640 avis (7.5%)

### Déséquilibre confirmé
Les avis positifs représentent 78.1% du dataset.
Ce déséquilibre sera corrigé dans le notebook suivant
avant d'entraîner le modèle.

### Les exemples sont cohérents
- Le POSITIF parle de produits de bonne qualité
- Le NEGATIF parle d'un produit mal étiqueté
- Le NEUTRE donne un avis mitigé

## 7. Sauvegarde des données

On sauvegarde uniquement les colonnes utiles (`Text` et `sentiment`)
dans le dossier `output/` pour les réutiliser dans le prochain notebook.

In [7]:
# Garder uniquement les colonnes utiles
df_final = df[["Text", "sentiment"]].copy()

# Sauvegarder dans le dossier output/
df_final.to_csv("../outputs/01_data_exploration.csv", index=False)

# Vérifier que le fichier est bien sauvegardé
df_verification = pd.read_csv("../outputs/01_data_exploration.csv")
print(f" Fichier sauvegardé avec succès !")
print(f"Lignes : {df_verification.shape[0]}")
print(f"Colonnes : {df_verification.columns.tolist()}")
print(f"\nAperçu :")
df_verification.head(3)

 Fichier sauvegardé avec succès !
Lignes : 568454
Colonnes : ['Text', 'sentiment']

Aperçu :


,Text,sentiment
0,I have bought several of the Vitality canned d...,positif
1,Product arrived labeled as Jumbo Salted Peanut...,negatif
2,This is a confection that has been around a fe...,positif


## Résumé de ce notebook

Dans ce notebook on a :

1. **Chargé** le dataset Amazon Food Reviews (568 454 avis)
2. **Exploré** les 10 colonnes et identifié les 2 utiles : `Text` et `Score`
3. **Vérifié** la qualité des données (0 valeurs manquantes, 0 doublons)
4. **Analysé** la répartition des notes (déséquilibre détecté : 78% positifs)
5. **Créé** les labels de sentiment à partir des notes
6. **Sauvegardé** le résultat dans `output/01_data_exploration.csv`

## Prochain notebook
`02_nettoyage.ipynb` — Nettoyage du texte :
- Supprimer la ponctuation et les majuscules
- Supprimer les mots inutiles (the, is, a...)
- Préparer le texte pour le modèle